# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cjalanano-dev/flyrank-ml-internship/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

**Lane:** Refresh / Content Opportunity Scoring (Lane 2)  
**Dataset:** FlyRank internship warehouse — `fact_content_daily_performance` (partitioned by month)  
**Development month:** `month=2026-03` (mid-panel; June 2026 is sealed test month)

> AI assistant note: skills loaded — `writing-data-contracts/SKILL.md` + `flyrank/flyrank-data/SKILL.md`

## 0. Setup — install DuckDB and authenticate Hugging Face

In [1]:
import subprocess, sys
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'duckdb', 'huggingface_hub'])

0

In [ ]:
import os
import duckdb
import pandas as pd

# Colab Secrets — never paste tokens inline; repo is public
try:
    from google.colab import userdata
    HF_TOKEN = userdata.get('HF_TOKEN')
except Exception:
    HF_TOKEN = os.environ.get('HF_TOKEN', '')

if not HF_TOKEN:
    raise EnvironmentError(
        'HF_TOKEN not found. '
        'In Colab: open the Secrets panel (key icon), add HF_TOKEN, then re-run. '
        'Locally: set the environment variable HF_TOKEN=<your-token>.'
    )

print('HF_TOKEN loaded (value hidden)')

In [ ]:
con = duckdb.connect()
con.execute('INSTALL httpfs; LOAD httpfs;')
con.execute(f"""
    SET s3_endpoint = 'huggingface.co';
    SET s3_access_key_id = 'user';
    SET s3_secret_access_key = '{HF_TOKEN}';
    SET s3_session_token = '';
    SET s3_url_style = 'path';
""")

BASE = 'hf://datasets/FlyRank/internship-warehouse'
DEV_MONTH = 'month=2026-03'
FACT = f'{BASE}/fact_content_daily_performance/{DEV_MONTH}/*.parquet'
DIM_CONTENT = f'{BASE}/dim_content/*.parquet'
DIM_CLIENTS = f'{BASE}/dim_clients/*.parquet'

print(f'DuckDB {duckdb.__version__} connected')
print(f'Dev partition: {DEV_MONTH}')

## 1. Unit of analysis + time window

### Five contract answers (plain words)

| # | Question | Answer |
|---|---|---|
| 1 | **What does one row mean?** | One row = one content item (page) for one client on one calendar day. It records that day's search and engagement signals — impressions, clicks, average position, sessions — from GSC and GA4. |
| 2 | **Which table(s)?** | Primary: `fact_content_daily_performance` (partitioned by `month`). Joined on demand to `dim_content` for content metadata and `dim_clients` for per-client history coverage dates. |
| 3 | **Which time window?** | Feature window: 90 days of daily rows ending at a chosen decision date. Label window: the 30 days immediately after that date. The two windows must not overlap — that is the leakage boundary. Development queries use `month=2026-03`. |
| 4 | **What do you predict or rank (label or proxy)?** | A page-level refresh priority score. Proxy label: a content item is **declining** if its impressions in the most recent 30 days of the feature window are ≥ 20% lower than in the 30 days before that (`impressions_last30 / impressions_prev30 < 0.80`), with a minimum of 100 prior-period impressions to exclude noise rows. |
| 5 | **What is deliberately excluded?** | Rows where `ga4_data_available IS NOT TRUE` — these carry zero-filled GA4 columns meaning 'not yet measured', not 'no engagement'. Including them injects false zeros into engagement features. Also excluded: the `_sample` partition (sealed June 2026 test month) and any column derived from the label's outcome window. |

In [ ]:
# Quick sanity-check: peek at fact table columns
schema = con.execute(f"DESCRIBE SELECT * FROM read_parquet('{FACT}') LIMIT 0").df()
print(f'Columns in fact table ({DEV_MONTH}): {len(schema)}')
print(schema[['column_name', 'column_type']].to_string(index=False))

## 2. Fields: feature / label / context / excluded

| Field | Bucket | Why |
|---|---|---|
| `report_date` | **Context** | Defines the daily row; used for window slicing, not a model input |
| `client_hash_id` | **Context** | Joins to `dim_clients`; for grouped train/test splits only |
| `content_hash_id` | **Context** | Joins to `dim_content`; grouping and deduplication only |
| `gsc_impressions` | **Feature** | Published ~2 days after the search date; available before any editorial decision |
| `gsc_clicks` | **Feature** | Same GSC availability window as impressions |
| `gsc_avg_position` | **Feature** | GSC position from the feature window; reflects pre-decision ranking strength |
| `sessions` | **Feature** | GA4 sessions; only used when `ga4_data_available IS TRUE` |
| `engaged_sessions` | **Feature** | GA4 engaged sessions; same availability guard |
| `sessions_ai` | **Feature** | AI-referred click-throughs; very sparse — used as a has-flag |
| `ga4_data_available` | **Context** | Filter gate; never a model feature |
| `gsc_data_available` | **Context** | Filter gate; same role |
| Future-window aggregates | **Label / Excluded** | Any aggregate of `gsc_impressions` from the 30-day window *after* the decision date — label definition, never a feature |
| `content_type` (via `dim_content`) | **Feature** | Static structural metadata; knowable before any decision |
| `content_created_at` (via `dim_content`) | **Feature** | Content age at decision time; purely historical |

**Proxy label:** `is_declining = (impressions_prev30 >= 100) AND (impressions_last30 / impressions_prev30 < 0.80)`  
This is within the current feature window (proxy). True capstone label uses a held-out future partition.

In [ ]:
# Verify that the availability flag columns exist
col_names = schema['column_name'].tolist()
for col in ['ga4_data_available', 'gsc_data_available']:
    tag = 'FOUND' if col in col_names else 'MISSING'
    print(f'  {tag}: {col}')

## 3. Verify with queries

Three small queries on `month=2026-03`. Each backs exactly one contract claim.

---

### Query 1 — Grain: one row really is `report_date × client × content`

**Claim:** no duplicate `(report_date, client_hash_id, content_hash_id)` combination exists.  
**Check:** zero rows returned = grain holds.

In [ ]:
q1 = (
    f"SELECT\n"
    f"    report_date,\n"
    f"    client_hash_id,\n"
    f"    content_hash_id,\n"
    f"    COUNT(*) AS row_count\n"
    f"FROM read_parquet('{FACT}')\n"
    f"GROUP BY report_date, client_hash_id, content_hash_id\n"
    f"HAVING COUNT(*) > 1\n"
    f"LIMIT 5"
)
result_q1 = con.execute(q1).df()
print(f'Query 1 — Grain check (month=2026-03)')
print(f'Rows with duplicate grain: {len(result_q1)}')
if len(result_q1) == 0:
    print('Grain holds: report_date x client_hash_id x content_hash_id is unique.')
else:
    print('Grain VIOLATED — inspect duplicates:')
    print(result_q1)

### Query 2 — Row count and date span for the March 2026 partition

**Claim:** March 2026 covers ~31 calendar days with a bounded, meaningful count of content-item-day rows.

In [ ]:
q2 = (
    f"SELECT\n"
    f"    COUNT(*)                            AS total_rows,\n"
    f"    COUNT(DISTINCT report_date)         AS distinct_dates,\n"
    f"    MIN(report_date)                    AS earliest_date,\n"
    f"    MAX(report_date)                    AS latest_date,\n"
    f"    COUNT(DISTINCT client_hash_id)      AS distinct_clients,\n"
    f"    COUNT(DISTINCT content_hash_id)     AS distinct_content_items\n"
    f"FROM read_parquet('{FACT}')"
)
result_q2 = con.execute(q2).df()
print('Query 2 — Row count and date span (month=2026-03)')
print(result_q2.to_string(index=False))

### Query 3 — Availability: how many rows survive `ga4_data_available IS TRUE`

**Claim:** rows where `ga4_data_available` is NULL or FALSE have zero-filled GA4 columns and must be excluded.  
**Note:** `IS TRUE` is required — the flag has three states (TRUE / FALSE / NULL). Writing `= TRUE` or `!= FALSE` silently mis-handles NULL rows, giving wrong counts.

In [ ]:
q3 = (
    f"SELECT\n"
    f"    COUNT(*)                                                   AS total_rows,\n"
    f"    SUM(CASE WHEN ga4_data_available IS TRUE  THEN 1 ELSE 0 END) AS ga4_available,\n"
    f"    SUM(CASE WHEN ga4_data_available IS FALSE THEN 1 ELSE 0 END) AS ga4_false,\n"
    f"    SUM(CASE WHEN ga4_data_available IS NULL  THEN 1 ELSE 0 END) AS ga4_null,\n"
    f"    ROUND(\n"
    f"        100.0 * SUM(CASE WHEN ga4_data_available IS TRUE THEN 1 ELSE 0 END)\n"
    f"              / COUNT(*), 1\n"
    f"    )                                                          AS pct_available\n"
    f"FROM read_parquet('{FACT}')"
)
result_q3 = con.execute(q3).df()
print('Query 3 — GA4 availability (month=2026-03, using IS TRUE)')
print(result_q3.to_string(index=False))
pct = result_q3['pct_available'].iloc[0]
print(f'\n{pct:.1f}% of rows have real GA4 data.'
      f' The rest must be excluded before using engagement features.')

## 4. Five features + the deliberate-leak experiment

### 4a. Five-feature frame

Aggregated from March 2026 daily rows to one row per content item.  
Every feature must be knowable before the editorial decision is made.

| Feature | Knowable at the decision moment because… |
|---|---|
| `log_impressions_90d` | GSC impressions are published with ~2-day lag; the full window sum is available before any content action. Log-transformed because traffic is heavy-tailed. |
| `avg_position_90d` | GSC average position averaged over the feature window; measures ranking strength before any decision. Rows where position = 0 (no data) are excluded from the average. |
| `ctr_90d` | clicks / impressions over the 90-day feature window; both numerator and denominator are entirely from the past. |
| `pct_days_with_impressions` | Fraction of feature-window days where the page had ≥ 1 impression; a consistency signal computed from past data only. |
| `log_sessions_90d` | GA4 sessions summed over the window, guarded with `ga4_data_available IS TRUE`. Pages without GA4 coverage contribute 0 and are flagged separately. |

**Label (proxy):** `is_declining` = impressions in most-recent half of month < 80% of prior half, with ≥ 50 impressions in the prior half as a noise guard.  
**Not included:** `trend_direction` / `trend_pct` — they encode the label and are never features.

In [ ]:
import numpy as np

# Build five-feature frame from March 2026, one row per content item
q_features = (
    f"WITH daily AS (\n"
    f"    SELECT content_hash_id, client_hash_id, report_date,\n"
    f"           gsc_impressions, gsc_clicks, gsc_avg_position,\n"
    f"           sessions, ga4_data_available, gsc_data_available\n"
    f"    FROM read_parquet('{FACT}')\n"
    f"    WHERE gsc_data_available IS TRUE\n"
    f"),\n"
    f"bounds AS (SELECT MAX(report_date) AS max_date FROM daily),\n"
    f"agg AS (\n"
    f"    SELECT\n"
    f"        d.content_hash_id,\n"
    f"        d.client_hash_id,\n"
    f"        LOG(1 + SUM(gsc_impressions))                    AS log_impressions_90d,\n"
    f"        AVG(CASE WHEN gsc_avg_position > 0 THEN gsc_avg_position END) AS avg_position_90d,\n"
    f"        CASE WHEN SUM(gsc_impressions) > 0\n"
    f"             THEN 100.0 * SUM(gsc_clicks) / SUM(gsc_impressions)\n"
    f"             ELSE NULL END                               AS ctr_90d,\n"
    f"        100.0 * SUM(CASE WHEN gsc_impressions > 0 THEN 1 ELSE 0 END)\n"
    f"              / COUNT(*)                                 AS pct_days_with_impressions,\n"
    f"        LOG(1 + SUM(CASE WHEN ga4_data_available IS TRUE THEN sessions ELSE 0 END)) AS log_sessions_90d,\n"
    f"        SUM(CASE WHEN report_date > (SELECT max_date FROM bounds) - INTERVAL 15 DAYS\n"
    f"                 THEN gsc_impressions ELSE 0 END)        AS impressions_last15,\n"
    f"        SUM(CASE WHEN report_date <= (SELECT max_date FROM bounds) - INTERVAL 15 DAYS\n"
    f"                 THEN gsc_impressions ELSE 0 END)        AS impressions_prev15\n"
    f"    FROM daily d\n"
    f"    GROUP BY d.content_hash_id, d.client_hash_id\n"
    f"    HAVING SUM(gsc_impressions) >= 10\n"
    f")\n"
    f"SELECT * FROM agg LIMIT 100000"
)

feat_df = con.execute(q_features).df()
print(f'Feature frame shape: {feat_df.shape}')
feature_cols = ['log_impressions_90d', 'avg_position_90d', 'ctr_90d',
                'pct_days_with_impressions', 'log_sessions_90d']
print('\nSample (5 rows):')
print(feat_df[feature_cols].head().to_string(index=False))

In [ ]:
# Compute proxy label
feat_df['is_declining'] = (
    (feat_df['impressions_prev15'] >= 50) &
    (feat_df['impressions_last15'] < 0.80 * feat_df['impressions_prev15'])
).astype(int)

n_total = len(feat_df)
n_dec   = feat_df['is_declining'].sum()
print(f'Feature frame rows   : {n_total:,}')
print(f'Declining (label=1)  : {n_dec:,} ({100*n_dec/n_total:.1f}%)')
print(f'Stable    (label=0)  : {n_total - n_dec:,}')
print('\nFeature summary stats:')
print(feat_df[feature_cols].describe().round(3).to_string())

### 4b. Deliberate-leak experiment

**The trap:** include a column derived from (or identical to) the label's inputs. The AUC jumps toward 1.0 — but no real signal was discovered.

We deliberately add `impressions_last15` (the direct numerator of the decline ratio) as a model feature, observe the inflated AUC, then **delete it** and re-score with only the five safe features.

This reproduces the leakage lesson from notebook 02, performed on real warehouse data.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score
from sklearn.preprocessing import StandardScaler
import warnings
warnings.filterwarnings('ignore')

model_df = feat_df[feature_cols + ['impressions_last15', 'is_declining']].dropna()
y = model_df['is_declining'].values

# --- LEAKED model: five features + impressions_last15 ---
X_leak = model_df[feature_cols + ['impressions_last15']].values
Xtr, Xte, ytr, yte = train_test_split(X_leak, y, test_size=0.2, random_state=42)
sc = StandardScaler()
clf_leak = LogisticRegression(max_iter=300).fit(sc.fit_transform(Xtr), ytr)
auc_leak = roc_auc_score(yte, clf_leak.predict_proba(sc.transform(Xte))[:, 1])

print('=' * 55)
print(' DELIBERATE LEAK -- includes impressions_last15')
print('=' * 55)
print(f' ROC-AUC (LEAKED) : {auc_leak:.4f}   <- suspiciously high!')
print()
print('Why? impressions_last15 is the direct numerator of the')
print('decline ratio that defines is_declining. The model reads')
print('the answer directly from its own input -- that is leakage.')

In [ ]:
# --- DELETE the leaked column --- honest model ---
X_hon = model_df[feature_cols].values
Xtr_h, Xte_h, ytr_h, yte_h = train_test_split(X_hon, y, test_size=0.2, random_state=42)
sc_h = StandardScaler()
clf_hon = LogisticRegression(max_iter=300).fit(sc_h.fit_transform(Xtr_h), ytr_h)
auc_hon = roc_auc_score(yte_h, clf_hon.predict_proba(sc_h.transform(Xte_h))[:, 1])

print('=' * 55)
print(' HONEST MODEL -- five safe features only')
print('=' * 55)
print(f' ROC-AUC (HONEST) : {auc_hon:.4f}')
print()
print(f' Gap (leak - honest): {auc_leak - auc_hon:+.4f}')
print()
print('The honest number is the real baseline.')
print('Any gain from here must come from better safe features.')

In [ ]:
print('Honest features used:')
for f in feature_cols:
    print(f'  - {f}')
print()
print('impressions_last15 -> DELETED (not in honest model)')

### Leak experiment summary

| Model | ROC-AUC | Honest? |
|---|---:|---|
| With `impressions_last15` (leaked) | see cell output | No — the answer was in the input |
| Five safe features only | see cell output | Yes — real signal only |

**Lesson:** the leaked AUC is a red flag, not a result. The honest number is where real modeling starts.

## 4 (cont.) — Data limits

### Named limitation: unbalanced history depth creates survivorship bias in window-based features

The daily fact table starts at `2025-01-27` at the earliest, but each client's tracking began on different dates (`dim_clients.gsc_data_start`). A content item published in October 2025 for a client who started tracking in December 2025 has only ~3 months of history, not the full 90 days the feature window assumes.

This means:
- `pct_days_with_impressions` is inflated for newer pages (smaller denominator).
- Volume aggregates mix pages with 90 days of evidence against pages with 12 days, making the model harder to calibrate.

**What this slice cannot tell us:** whether a page's signal is genuinely stable or only looks stable because there are very few days of data for it. Capstone work should join `dim_clients.gsc_data_start` and filter to content items with at least 60 days of observable history before building any window-based feature.

In [ ]:
# Show spread of days-observed per content item in this partition
q_limit = (
    f"SELECT days_observed, COUNT(*) AS content_items\n"
    f"FROM (\n"
    f"    SELECT content_hash_id, COUNT(DISTINCT report_date) AS days_observed\n"
    f"    FROM read_parquet('{FACT}')\n"
    f"    WHERE gsc_data_available IS TRUE\n"
    f"    GROUP BY content_hash_id\n"
    f") t\n"
    f"GROUP BY days_observed\n"
    f"ORDER BY days_observed\n"
    f"LIMIT 10"
)
lim_df = con.execute(q_limit).df()
print('Days observed per content item (March 2026) -- first 10 buckets:')
print(lim_df.to_string(index=False))
print()
print('Items with fewer days have less reliable window-based features.')

## 5. Self-check

Before you submit, confirm each line honestly:

- [x] **Five plain-words contract answers** — filled in Section 1 (one-row meaning, tables, time window, label/proxy, exclusion)
- [x] **Exactly three verification queries** — Query 1 (grain via `HAVING COUNT(*) > 1`), Query 2 (row count + date span), Query 3 (availability via `IS TRUE`)
- [x] **Five-feature frame** — built in Section 4a with one 'knowable when?' line per feature
- [x] **Deliberate-leak experiment** — `impressions_last15` added, AUC inflated, column deleted, honest AUC kept
- [x] **One named limitation** — unbalanced history depth / survivorship bias in window-based features
- [x] Notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] Claims use careful words: observed, proxy, measured, directional, decision-support
- [x] Committed to repo under `work/notebooks/` — then submit repo URL on the card. Done.